# 필요 라이브러리

In [56]:
import os
import re

import pandas as pd
import numpy as np

# 데이터 로드

In [57]:
DATA_PATH = os.path.join('data', 'draft8_offline.csv')

df_raw = pd.read_csv(DATA_PATH, encoding='utf-8-sig')
df_raw.head()

,결제일시,결제일,결제년월,결제년,결제월,결제시간,결제시간대,요일,카테고리,수량,상품별 단가,상품명_추출,Hot/Ice,원두대분류,원두종류,원두용량,할인여부,취식 방법,매출
0,2022-02-10 10:03:28,2022-02-10,2022-02,2022,2,10:03:28,10,목요일,핸드드립,1,10000,과테말라 레드 파카마라,Hot,싱글,과테말라 레드 파카마라,-,-,-,10000
1,2022-02-10 10:03:28,2022-02-10,2022-02,2022,2,10:03:28,10,목요일,핸드드립,1,10500,콜롬비아 로꼬 소르베,Ice,싱글,콜롬비아 로꼬 소르베,-,-,-,10500
2,2022-02-10 10:03:28,2022-02-10,2022-02,2022,2,10:03:28,10,목요일,핸드드립,1,12000,니카라과 COE#1,Hot,싱글,니카라과 COE#1,-,-,-,12000
3,2022-02-10 10:13:57,2022-02-10,2022-02,2022,2,10:13:57,10,목요일,에스프레소,1,6500,슈퍼클린에스프레소,Hot,블렌딩,클래식,-,-,-,6500
4,2022-02-10 10:13:57,2022-02-10,2022-02,2022,2,10:13:57,10,목요일,에스프레소,1,6500,슈퍼클린에스프레소,Hot,블렌딩,쥬시,-,-,-,6500


In [58]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 153433 entries, 0 to 153432
Data columns (total 19 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   결제일시     153433 non-null  object
 1   결제일      153433 non-null  object
 2   결제년월     153433 non-null  object
 3   결제년      153433 non-null  int64 
 4   결제월      153433 non-null  int64 
 5   결제시간     153433 non-null  object
 6   결제시간대    153433 non-null  int64 
 7   요일       153433 non-null  object
 8   카테고리     153433 non-null  object
 9   수량       153433 non-null  int64 
 10  상품별 단가   153433 non-null  int64 
 11  상품명_추출   153433 non-null  object
 12  Hot/Ice  153433 non-null  object
 13  원두대분류    153433 non-null  object
 14  원두종류     153433 non-null  object
 15  원두용량     153433 non-null  object
 16  할인여부     153433 non-null  object
 17  취식 방법    153433 non-null  object
 18  매출       153433 non-null  int64 
dtypes: int64(6), object(13)
memory usage: 22.2+ MB


# 전처리

In [59]:
df = df_raw.copy()

## 기본 정리

In [60]:
text_cols = df.select_dtypes(include='object').columns

df[text_cols] = df[text_cols].apply(lambda col: col.str.strip())
df[text_cols] = df[text_cols].replace('', np.nan)

In [61]:
# 중복 행 확인
df.duplicated().sum()

np.int64(32)

## 날짜/시간 컬럼

In [62]:
df['결제일시'] = pd.to_datetime(df['결제일시'], errors='coerce')
df['결제일'] = pd.to_datetime(df['결제일'], errors='coerce')

df['결제년월'] = df['결제일'].dt.to_period('M').astype(str)
df['결제년'] = df['결제일'].dt.year.astype('Int64')
df['결제월'] = df['결제일'].dt.month.astype('Int64')
df['결제시간'] = df['결제일시'].dt.strftime('%H:%M:%S')
df['결제시간대'] = df['결제일시'].dt.hour.astype('Int64')

In [63]:
weekday_map = {
    0: '월요일',
    1: '화요일',
    2: '수요일',
    3: '목요일',
    4: '금요일',
    5: '토요일',
    6: '일요일',
}

df['요일'] = df['결제일'].dt.dayofweek.map(weekday_map)

## 숫자 컬럼

In [64]:
numeric_cols = ['수량', '상품별 단가', '매출']

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df[numeric_cols].isna().sum()

수량        0
상품별 단가    0
매출        0
dtype: int64

In [65]:
# 매출 계산 검증용 컬럼
df['매출_계산값'] = df['수량'] * df['상품별 단가']
df['매출_차이'] = df['매출'] - df['매출_계산값']

df.loc[df['매출_차이'] != 0, ['수량', '상품별 단가', '매출', '매출_계산값', '매출_차이']].head()

,수량,상품별 단가,매출,매출_계산값,매출_차이


## 카테고리/상품명 전처리

In [66]:
# 문자열 전처리 공통 함수
# - normalize_text: 앞뒤 공백/이모지/연속 공백 정리
# - compact_text: 상품명 비교용으로 모든 공백 제거
# - underscore_text: 최종 상품명 표기용으로 공백을 '_'로 변경
def normalize_text(value):
    if pd.isna(value):
        return ''

    value = str(value).strip()
    value = re.sub(r'[☕🟢🟤]', '', value)
    value = re.sub(r'\s+', ' ', value)
    return value.strip()


def compact_text(value):
    return re.sub(r'\s+', '', normalize_text(value))


def underscore_text(value):
    value = normalize_text(value)
    value = re.sub(r'\s+', '_', value)
    value = re.sub(r'_+', '_', value)
    return value.strip('_')


# Hot/Ice 값을 참고파일의 최종 상품명 접두사인 H/I로 변환
def temperature_code(value):
    value = normalize_text(value).lower()
    if value.startswith('ice') or value == 'i':
        return 'I'
    if value.startswith('hot') or value == 'h':
        return 'H'
    return ''


# 원두명 표기를 정리하고, 싱글오리진은 참고파일 결과와 맞춰 싱글로 통일
def bean_name(value):
    value = normalize_text(value)
    value = re.sub(r'^-$', '', value)
    value = re.sub(r'싱글오리진', '싱글', value)
    return value

In [67]:
# 카테고리 전처리
# 참고파일의 수작업 규칙을 정규표현식으로 묶어 기존 카테고리 컬럼에 반영
def preprocess_category(row):
    category = normalize_text(row['카테고리'])
    product = normalize_text(row['상품명_추출'])

    # 종이백/캐리어/보냉백은 판매 상품이 아니라 포장 카테고리로 분리
    if re.fullmatch(r'(종이백|캐리어|보냉백)', product):
        return '포장'
    # 포장/야외/실내는 메뉴가 아니라 주문 유형으로 분리
    if re.fullmatch(r'(포장|야외|실내)', product):
        return '주문유형'
    # 분쇄 옵션은 원두 상품에서 별도 분쇄 카테고리로 분리
    if re.search(r'^분쇄\s*_\s*(핸드드립|모카모트|모카포트)$', product):
        return '분쇄'
    # 샷/시럽/얼음/우유 옵션류는 커스텀 카테고리로 분리
    if re.fullmatch(r'(샷추가|덜 달게|오틀리|연하게|1샷 추가|2샷 추가|시럽|바닐라시럽|얼음 적게|물적게|얼음 X)', product):
        return '커스텀'
    # 무료시음권/선결제/상품권/원두 별도구매는 분석용 메뉴 카테고리에서 제외
    if re.fullmatch(r'(무료시음권|선결제|상품권\s*3만원권|원두 별도구매)', product):
        return '미분류'
    # 사업자 카테고리에 섞인 공연용 시그니처 메뉴는 시그니처로 이동
    if re.fullmatch(r'공연\s*(텐저린카푸치노|아이스텐라|아이스텐저린라떼)', product):
        return '시그니처'
    # 참고파일의 Basic _ ice 결과와 맞추기 위해 Basic 중 Ice 메뉴는 Basic_ice로 분리
    if category == 'Basic' and temperature_code(row['Hot/Ice']) == 'I':
        return 'Basic_ice'
    if category == 'Basic _ ice':
        return 'Basic_ice'

    return category

In [68]:
# 커피류 상품명 전처리
# 최종 형태: (H/I)_원두_상품명, 세트인 경우 뒤에 _세트 추가
def coffee_product_name(row, suffix=''):
    product = compact_text(row['상품명_추출'])
    bean = bean_name(row['원두종류'])
    temp = temperature_code(row['Hot/Ice'])

    # 공연용 시그니처 메뉴는 참고파일처럼 일반 시그니처 메뉴명으로 통일
    if re.fullmatch(r'공연\s*텐저린카푸치노', normalize_text(row['상품명_추출'])):
        product, bean, temp = '텐저린카푸치노', '쥬시', 'H'
    if re.fullmatch(r'공연\s*(아이스텐라|아이스텐저린라떼)', normalize_text(row['상품명_추출'])):
        product, bean, temp = '텐저린라떼', '쥬시', 'I'
    # 미미MIMI는 원두/온도 조합 규칙이 아닌 고정 상품명으로 처리
    if product == '미미MIMI':
        return '(I)_미미MIMI'
    # 에스프레소 프레도는 참고파일에서 Ice 상품으로 처리
    if re.search(r'프레도', product):
        temp = 'I'
    # 슈퍼클린에스프레소/카페루이지에스프레소는 뒤의 에스프레소 중복 표현 제거
    if re.fullmatch(r'(슈퍼클린|카페루이지)\s*에스프레소', product):
        product = re.sub(r'에스프레소$', '', product)
    # 루이지/블렌딩처럼 참고파일에서 쥬시로 정리된 원두 표기 통일
    if bean == '루이지':
        bean = '쥬시'
    if bean == '블렌딩' and product != '미미MIMI':
        bean = '쥬시'

    parts = [part for part in [f'({temp})' if temp else '', bean, product] if part]
    return '_'.join(parts) + suffix

In [69]:
# 디저트 상품명 전처리
# 공백 없는 상품명을 참고파일 결과처럼 주요 단어 사이에 '_'를 넣어 정리
def dessert_product_name(product):
    product = compact_text(product)
    product = re.sub(r'^⚪️?$', '무화과휘낭시에', product)
    product = re.sub(r'(바스크)(치즈케이크)', r'_', product)
    product = re.sub(r'(플레인|무화과|레몬|시나몬|헤이즐넛)(휘낭시에)', r'_', product)
    return product


# 비버리지 상품명 전처리
# 최종 형태: (H/I)_지역_재료_상품명 또는 (H/I)_상품명
def beverage_product_name(row):
    product = compact_text(row['상품명_추출'])
    temp = temperature_code(row['Hot/Ice'])

    # 감귤주스는 참고파일 결과에 맞춰 귤피주스로 통일
    product = re.sub(r'^제주유기농감귤주스$', '제주유기농귤피주스', product)
    # 지역/재료/음료 종류가 붙어 있는 상품명은 '_'로 구조화
    product = re.sub(r'^(제주)(유기농)(귤피주스)$', r'__', product)
    product = re.sub(r'^(문경)(선암리)(사과주스)$', r'__', product)
    product = re.sub(r'^(패션프루트|천혜향|오미자|유자)(에이드)$', r'_', product)
    product = re.sub(r'^(얼그레이)(밀크티)$', r'_', product)
    product = re.sub(r'^(거제)(유기농)(유자차)$', r'__', product)
    product = re.sub(r'^(문경)(오미자차)$', r'_', product)
    product = re.sub(r'^(트로피칼)(루이보스)$', r'_차', product)
    product = re.sub(r'^(시나몬)(플럼)$', r'_차', product)
    product = re.sub(r'^(카모마일)$', r'차', product)
    product = re.sub(r'^(오미자차)$', '문경_오미자차', product)
    product = re.sub(r'^(유기농유자차)$', '거제_유기농_유자차', product)

    # Hot/Ice 정보가 있으면 참고파일처럼 상품명 앞에 온도 접두사 추가
    return f'({temp})_{product}' if temp else product


# 드립백/캡슐 상품명 전처리
# 캡슐, 드립백, 세트, ea 수량 정보를 정규식으로 추출해 참고파일 표기로 변환
def drip_capsule_product_name(row):
    product = normalize_text(row['상품명_추출'])
    bean = bean_name(row['원두종류'])

    # 대용량 캡슐은 별도 상품명으로 처리
    if re.search(r'캡슐대용량', product):
        return '캡슐_대용량'
    # 1+1 캡슐은 [1+1] 접두사와 원두명을 결합
    if re.search(r'1\s*\+\s*1\s*캡슐', product):
        return f'[1+1]캡슐_{compact_text(bean)}' if bean else '[1+1]캡슐'
    # 일반 캡슐은 원두명을 뒤에 붙여 캡슐_원두 형태로 정리
    if re.fullmatch(r'캡슐', product):
        return f'캡슐_{compact_text(bean)}' if bean else '캡슐'
    # 샘플러/컬렉션/세트는 참고파일의 고정 상품명 규칙 적용
    if re.search(r'샘플러\s*4종', product):
        return '드립백_샘플러4종'
    if re.search(r'컬렉션', product):
        return '드립백_컬렉션'
    if re.search(r'세트', product):
        return f'드립백_{compact_text(bean)}세트' if bean else '드립백_세트'

    # 2ea/3ea/5ea/10ea 등 수량 표기는 원두명 뒤에 붙여 정리
    ea_match = re.search(r'(\d+)\s*ea', product, flags=re.IGNORECASE)
    if ea_match:
        ea = ea_match.group(1)
        return f'드립백_{compact_text(bean)}{ea}ea' if bean else f'드립백_{ea}ea'
    if re.fullmatch(r'드립백', product):
        return f'드립백_{compact_text(bean)}' if bean else '드립백'

    return underscore_text(product)

In [70]:
# 핸드드립 상품명 전처리
# 최종 형태: (H/I)_원두명, 디카페인 원두는 상품명에 디카프 표기를 보강
def handdrip_product_name(row):
    product = normalize_text(row['상품명_추출'])
    temp = temperature_code(row['Hot/Ice'])

    # 디카페인 핸드드립인데 상품명에 디카프가 없으면 참고파일 결과처럼 디카프 추가
    if normalize_text(row['원두대분류']) == '디카페인' and not re.search(r'디카프', product):
        product = f'{product} 디카프'

    product = underscore_text(product)
    return f'({temp})_{product}' if temp else product


# 원두 상품명 전처리
# 최종 형태: 카테고리_원두명_용량, 대용량/할인 여부는 접미·접두사로 반영
def bean_product_name(row):
    category = row['카테고리']
    bean = bean_name(row['원두종류']) or normalize_text(row['상품명_추출'])
    volume = normalize_text(row['원두용량'])

    # 원두종류에 섞인 용량 표기는 제거하고 원두용량 컬럼을 우선 사용
    bean = re.sub(r'\b(\d{2,3}g|1kg|500g)\b', '', bean).strip()
    if not volume or volume == '-':
        # 원두용량이 비어 있으면 원두종류에서 용량을 재추출, 없으면 참고파일 기본값 200g 적용
        volume_match = re.search(r'\b(\d{2,3}g|1kg|500g)\b', normalize_text(row['원두종류']))
        volume = volume_match.group(1) if volume_match else '200g'

    name = f'{category}_{compact_text(bean)}_{volume}'
    # 클래식/쥬시 500g, 1kg는 참고파일처럼 대용량 표시 추가
    if category == '블렌딩원두' and re.fullmatch(r'(클래식|쥬시)', compact_text(bean)) and volume in ['500g', '1kg']:
        name += '_대용량'
    # 할인 원두는 [할인] 접두사 추가
    if str(row.get('할인여부', '')).lower() == 'true':
        name = '[할인]' + name
    return name


# MD 상품명 전처리
# 상품군에 따라 (음료)/(도서)/(의류)/(잡화) 접두사를 붙임
def md_product_name(product):
    product = compact_text(product)

    # RTD/오틀리/오트사이드는 음료 상품으로 분류
    if re.search(r'^(RTD텐라보틀|오틀리초코|오트사이드_?초코|오트사이드_?바리스타|오트사이드)$', product):
        product = re.sub(r'^오트사이드_?초코$', '오트사이트초코', product)
        product = re.sub(r'^오트사이드_?바리스타$', '오트사이드바리스타', product)
        return f'(음료)_{product}'
    # 도서류는 (도서) 접두사 추가
    if re.search(r'^(스페셜티커피|뮤직포시티트래블러|뮤직포레잇모닝)$', product):
        return f'(도서)_{product}'
    # KCW 의류 상품은 (의류) 접두사 추가
    if re.search(r'^\(KCW\)캠프캡$|^\(KCW\)티셔츠$', product):
        return f'(의류)_{re.sub(r"[() ]", "", product)}'
    # 스티커는 상품명에서 '스티커'를 제거하고 잡화로 처리
    if re.search(r'^(스티커제주|스티커동백|스티커캐릭터)$', product):
        return '(잡화)_' + re.sub(r'^스티커', '', product)

    return f'(잡화)_{re.sub(r"[() ]", "", product)}'


# 사업자 상품명 전처리
# 사업자/소량 접두 표현을 제거한 뒤 사업자_상품명 형태로 정리
def business_product_name(row):
    product = normalize_text(row['상품명_추출'])

    # 사업자에 섞인 공연 메뉴는 시그니처 상품명 결과와 동일하게 처리
    if re.fullmatch(r'공연\s*텐저린카푸치노', product):
        return '(H)_쥬시_텐저린카푸치노'
    if re.fullmatch(r'공연\s*(아이스텐라|아이스텐저린라떼)', product):
        return '(I)_쥬시_텐저린라떼'

    # [소량], 사업자 등의 접두 표현은 제거하고 '_'로 연결
    product = re.sub(r'^\[소량\]\s*', '', product)
    product = re.sub(r'^사업자\s*', '', product)
    product = underscore_text(product)
    return f'사업자_{product}'

In [71]:
# 상품명 전처리 라우터
# 전처리된 카테고리 값을 기준으로 카테고리별 정규식 전처리 함수를 선택
def preprocess_product_name(row):
    category = row['카테고리']
    product = normalize_text(row['상품명_추출'])

    # 커피류는 공통적으로 (H/I)_원두_상품명 구조 사용
    if category in ['Basic', 'Basic_ice', '시그니처', '에스프레소']:
        return coffee_product_name(row)
    # 세트 메뉴는 커피류 규칙에 _세트 접미사를 붙임
    if category == '세트':
        return coffee_product_name(row, suffix='_세트').replace('Set.', '')
    # 비커피류/상품류는 카테고리별 함수로 분기
    if category == '디저트':
        return dessert_product_name(product)
    if category == '비버리지':
        return beverage_product_name(row)
    if category == '핸드드립':
        return handdrip_product_name(row)
    if category == '드립백/캡슐':
        return drip_capsule_product_name(row)
    if category in ['싱글원두', '블렌딩원두']:
        return bean_product_name(row)
    if category == 'MD':
        return md_product_name(product)
    if category == '사업자':
        return business_product_name(row)
    # 분쇄/커스텀은 단순 표기 정리만 수행
    if category == '분쇄':
        return re.sub(r'\s*_\s*', '_', product)
    if category == '커스텀':
        return compact_text(product)

    return underscore_text(product)

In [72]:
# 카테고리와 상품명 전처리 결과를 기존 컬럼에 덮어쓰기
# 카테고리를 먼저 덮어쓴 뒤, 전처리된 카테고리를 기준으로 상품명_추출을 다시 생성
df['카테고리'] = df.apply(preprocess_category, axis=1)
df['상품명_추출'] = df.apply(preprocess_product_name, axis=1)

df[['카테고리', '상품명_추출']].head()

,카테고리,상품명_추출
0,핸드드립,(H)_과테말라_레드_파카마라
1,핸드드립,(I)_콜롬비아_로꼬_소르베
2,핸드드립,(H)_니카라과_COE#1
3,에스프레소,(H)_클래식_슈퍼클린
4,에스프레소,(H)_쥬시_슈퍼클린


In [73]:
df['카테고리'].value_counts()

카테고리
시그니처         44844
Basic_ice    22882
디저트          16771
에스프레소        14384
Basic        14128
비버리지         12176
핸드드립         10807
드립백/캡슐        7816
블렌딩원두         3835
싱글원두          3748
MD            1365
세트             434
분쇄             125
포장              74
사업자             42
미분류              2
Name: count, dtype: int64

In [74]:
df['상품명_추출'].value_counts().head(30)

상품명_추출
(I)_쥬시_텐저린라떼      21528
(I)_쥬시_유자아메리카노    12050
_               10396
(H)_쥬시_텐저린카푸치노     9541
(I)_클래식_아메리카노      9196
(H)_클래식_슈퍼클린       5073
(I)___          4911
(H)_클래식_아메리카노      4363
브라우니               4190
(H)_쥬시_슈퍼클린        4094
(I)__            3888
(I)_쥬시_아메리카노       3755
(H)_쥬시_카페루이지       3011
(I)_클래식_카페라떼       2547
(H)_클래식_카페라떼       2062
잠봉뵈르               2059
(I)_클래식_플랫화이트      1852
(H)_싱글_슈퍼클린        1773
드립백_클래식            1764
(H)_쥬시_아메리카노       1702
블렌딩원두_쥬시_200g      1282
(I)_싱글_아메리카노       1248
드립백_쥬시             1246
블렌딩원두_클래식_200g     1224
(H)_클래식_플랫화이트      1197
드립백_샘플러4종          1161
(I)_디카프_아메리카노      1056
(H)__             980
(I)_디카프_텐저린라떼       918
캡슐_클래식              838
Name: count, dtype: int64

## 정렬 및 확인

In [75]:
df = df.sort_values(['결제일시', '카테고리', '상품명_추출']).reset_index(drop=True)
df.head()

,결제일시,결제일,결제년월,결제년,결제월,결제시간,결제시간대,요일,카테고리,수량,...,상품명_추출,Hot/Ice,원두대분류,원두종류,원두용량,할인여부,취식 방법,매출,매출_계산값,매출_차이
0,2022-02-10 10:03:28,2022-02-10,2022-02,2022,2,10:03:28,10,목요일,핸드드립,1,...,(H)_과테말라_레드_파카마라,Hot,싱글,과테말라 레드 파카마라,-,-,-,10000,10000,0
1,2022-02-10 10:03:28,2022-02-10,2022-02,2022,2,10:03:28,10,목요일,핸드드립,1,...,(H)_니카라과_COE#1,Hot,싱글,니카라과 COE#1,-,-,-,12000,12000,0
2,2022-02-10 10:03:28,2022-02-10,2022-02,2022,2,10:03:28,10,목요일,핸드드립,1,...,(I)_콜롬비아_로꼬_소르베,Ice,싱글,콜롬비아 로꼬 소르베,-,-,-,10500,10500,0
3,2022-02-10 10:13:57,2022-02-10,2022-02,2022,2,10:13:57,10,목요일,시그니처,1,...,(I)_쥬시_텐저린라떼,Ice,블렌딩,쥬시,-,-,-,7000,7000,0
4,2022-02-10 10:13:57,2022-02-10,2022-02,2022,2,10:13:57,10,목요일,에스프레소,1,...,(H)_쥬시_슈퍼클린,Hot,블렌딩,쥬시,-,-,-,6500,6500,0


In [76]:
summary = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing_count': df.isna().sum(),
    'missing_rate': df.isna().mean().round(4),
    'nunique': df.nunique(dropna=True),
})

summary

,dtype,missing_count,missing_rate,nunique
결제일시,datetime64[ns],0,0.0,74108
결제일,datetime64[ns],0,0.0,517
결제년월,object,0,0.0,18
결제년,Int64,0,0.0,2
결제월,Int64,0,0.0,12
결제시간,object,0,0.0,27729
결제시간대,Int64,0,0.0,13
요일,object,0,0.0,7
카테고리,object,0,0.0,16
수량,int64,0,0.0,15


In [77]:
df.columns

Index(['결제일시', '결제일', '결제년월', '결제년', '결제월', '결제시간', '결제시간대', '요일', '카테고리',
       '수량', '상품별 단가', '상품명_추출', 'Hot/Ice', '원두대분류', '원두종류', '원두용량', '할인여부',
       '취식 방법', '매출', '매출_계산값', '매출_차이'],
      dtype='object')

In [78]:
df

,결제일시,결제일,결제년월,결제년,결제월,결제시간,결제시간대,요일,카테고리,수량,...,상품명_추출,Hot/Ice,원두대분류,원두종류,원두용량,할인여부,취식 방법,매출,매출_계산값,매출_차이
0,2022-02-10 10:03:28,2022-02-10,2022-02,2022,2,10:03:28,10,목요일,핸드드립,1,...,(H)_과테말라_레드_파카마라,Hot,싱글,과테말라 레드 파카마라,-,-,-,10000,10000,0
1,2022-02-10 10:03:28,2022-02-10,2022-02,2022,2,10:03:28,10,목요일,핸드드립,1,...,(H)_니카라과_COE#1,Hot,싱글,니카라과 COE#1,-,-,-,12000,12000,0
2,2022-02-10 10:03:28,2022-02-10,2022-02,2022,2,10:03:28,10,목요일,핸드드립,1,...,(I)_콜롬비아_로꼬_소르베,Ice,싱글,콜롬비아 로꼬 소르베,-,-,-,10500,10500,0
3,2022-02-10 10:13:57,2022-02-10,2022-02,2022,2,10:13:57,10,목요일,시그니처,1,...,(I)_쥬시_텐저린라떼,Ice,블렌딩,쥬시,-,-,-,7000,7000,0
4,2022-02-10 10:13:57,2022-02-10,2022-02,2022,2,10:13:57,10,목요일,에스프레소,1,...,(H)_쥬시_슈퍼클린,Hot,블렌딩,쥬시,-,-,-,6500,6500,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
153428,2023-07-31 16:59:01,2023-07-31,2023-07,2023,7,16:59:01,16,월요일,시그니처,1,...,(I)_쥬시_텐저린라떼,Ice,블렌딩,쥬시,-,-,-,7000,7000,0
153429,2023-07-31 17:17:57,2023-07-31,2023-07,2023,7,17:17:57,17,월요일,시그니처,3,...,(I)_쥬시_텐저린라떼,Ice,블렌딩,쥬시,-,-,-,21000,21000,0
153430,2023-07-31 17:19:12,2023-07-31,2023-07,2023,7,17:19:12,17,월요일,시그니처,1,...,(I)_쥬시_유자아메리카노,Ice,블렌딩,쥬시,-,-,-,7000,7000,0
153431,2023-07-31 17:19:12,2023-07-31,2023-07,2023,7,17:19:12,17,월요일,시그니처,1,...,(I)_쥬시_텐저린라떼,Ice,블렌딩,쥬시,-,-,-,7000,7000,0
